<a href="https://colab.research.google.com/github/Lolla-data-analyst/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# W03 Data Contract — Google Search Ranking & Discoverability Lane

## 1. Data Contract

**One row means:** One piece of content's search performance, for one client, on one specific day.

**Table(s) used:**
- `fact_content_daily_performance` (main table)
- `dim_clients` — for `is_active`, to filter out inactive clients
- `dim_content` — for `content_updated_date`, to check content freshness against performance

**Time window:** A mid-panel month, March 2026 (`report_date` between 2026-03-01 and 2026-03-31), deliberately avoiding the sealed final month (`_sample` / June 2026), so there's future data available to eventually validate against.

**What I'd rank:** Content, sorted by `gsc_avg_position` (lower = better). I'd compare the top-ranked group against the bottom-ranked group — not just single best/worst examples — to see what other signals differ between them.

**What I exclude:** Rows where `gsc_data_available = false`. If missing GSC data defaults `gsc_avg_position` to 0, those rows would falsely appear as the *best*-ranked content (since lower = better), contaminating the top-ranked group with fake results.

## 2. Verification Queries

**Grain check:** Grouped by `report_date`, `client_hash_id`, `content_hash_id` — every group had exactly 1 row. No duplicates found; the grain claim holds.

**Row count & date span:** 3,611,061 rows survive the availability filter, spanning 2026-03-01 to 2026-03-31 — a full, clean month.

**Availability impact:** Of 9,841,378 total rows in March, only 3,611,061 (~37%) have `gsc_data_available = true`. The exclusion rule removes the majority of rows, confirming it's a meaningful, not cosmetic, decision.

## 3. Feature Frame (5 features)

| Feature | Knowable at decision moment because... |
|---|---|
| `scroll_events` | Recorded in the same daily row as ranking data, no reporting lag |
| `sessions_ai` | Same-row availability; genuinely separate from Google ranking (different discovery mechanism) |
| Days since content update (past-only rule) | Only counted when `content_updated_date` precedes `report_date`, avoiding future-information leakage |
| Sessions-per-user ratio | Same-row availability; revisit behavior reflects content value, not search visibility |
| `sessions_referral` | Same-row availability; potential contributing factor to ranking, though the data can't distinguish organic vs. paid/reciprocal referral arrangements — interpreted cautiously |

**Note on coverage:** Only ~18% of rows (648,576 of 3,611,061) have a usable value for "days since update" in March 2026, since most content's `content_updated_date` falls after this month. The feature remains valid and leakage-free, but sparse for this window.

## 4. The Leakage Trap

**What I tested:** `gsc_sum_position`, a column that sounded like it could be a useful feature.

**What I found:** A simple correlation with `gsc_avg_position` was misleadingly low (0.116), since `gsc_sum_position` is also entangled with `gsc_impressions`. A direct check proved the real story: `gsc_sum_position ÷ gsc_impressions` equals `gsc_avg_position` exactly, with zero difference across all 3.6M+ rows in the slice.

**Why it's leakage:** `gsc_sum_position` is mathematically derived from the same information as the ranking target itself. Including it wouldn't add real insight — it would let any comparison "cheat" by seeing a disguised version of the answer. It was excluded from the final feature set.

In [1]:
%pip install -q duckdb huggingface_hub

In [2]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

print("Connected successfully")

Connected successfully


In [3]:
con.execute(f"CREATE OR REPLACE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

In [4]:
rel = "hf://datasets/FlyRank/internship-warehouse"

test = con.sql(f"SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') LIMIT 5")
test.show()

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_cl

In [5]:
result = con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS row_count
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
GROUP BY report_date, client_hash_id, content_hash_id
ORDER BY row_count DESC
LIMIT 10
""")
result.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬───────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ row_count │
│    date     │         varchar         │         varchar          │   int64   │
├─────────────┼─────────────────────────┼──────────────────────────┼───────────┤
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_d0dff76c889de68f │         1 │
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_67741cce996cfafa │         1 │
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_2e6360ad20fd7107 │         1 │
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_ac8663da7484669a │         1 │
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_65c50dfe9d87a585 │         1 │
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_d49a012dcb924e31 │         1 │
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_614baf2af4330bd7 │         1 │
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_4dc944b7d0b65ecc │         1 │
│ 2026-03-01  │ client_62f4a

In [6]:
result = con.sql(f"""
SELECT MIN(report_date), MAX(report_date), Count(*) AS row_count
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01' AND gsc_data_available = true
""")
result.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬──────────────────┬───────────┐
│ min(report_date) │ max(report_date) │ row_count │
│       date       │       date       │   int64   │
├──────────────────┼──────────────────┼───────────┤
│ 2026-03-01       │ 2026-03-31       │   3611061 │
└──────────────────┴──────────────────┴───────────┘



In [7]:
result = con.sql(f"""
SELECT COUNT(*) AS total_rows,
COUNT(CASE WHEN gsc_data_available = true THEN 1 ELSE NULL END) AS available_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""")
result.show()

┌────────────┬────────────────┐
│ total_rows │ available_rows │
│   int64    │     int64      │
├────────────┼────────────────┤
│    9841378 │        3611061 │
└────────────┴────────────────┘



In [8]:
result = con.sql(f"""
SELECT CORR(gsc_avg_position, gsc_sum_position)
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""")
result.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────────────────────┐
│ corr(gsc_avg_position, gsc_sum_position) │
│                  double                  │
├──────────────────────────────────────────┤
│                      0.11613006030170202 │
└──────────────────────────────────────────┘



In [9]:
result = con.sql(f"""
SELECT MAX(ABS(gsc_avg_position - gsc_sum_position / NULLIF(gsc_impressions, 0))) AS max_difference
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""")
result.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ max_difference │
│     double     │
├────────────────┤
│            0.0 │
└────────────────┘



The trap: I deliberately considered adding gsc_sum_position as a feature. A correlation check alone was misleading (only 0.116, since impressions vary independently), but a direct check proved the real story: gsc_sum_position ÷ gsc_impressions equals gsc_avg_position exactly, for every row in the slice (max difference = 0.0). This means gsc_sum_position is mathematically derived from the exact same information as my ranking target — including it wouldn't add real insight, it would just let the "prediction" cheat by seeing a disguised version of the answer. I excluded it and kept my honest 5-feature set instead.

In [12]:
from huggingface_hub import list_repo_files

files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=userdata.get('HF_TOKEN'))
dim_content_files = [f for f in files if 'dim_content' in f]
print(dim_content_files)

['dim_content.parquet']


In [13]:
test_dim = con.sql(f"SELECT * FROM read_parquet('{rel}/dim_content.parquet') LIMIT 3")
test_dim.show()

┌─────────────────────────┬──────────────────────────┬──────────────────────────┬──────────────────────┬────────────────────┬─────────────────────┬────────────────┬──────────────────────┬──────────────────────┬─────────────────┬───────────────┬─────────────┬───────────────────┬────────┬───────────────┬───────────┬────────────────┬──────────────────────┬─────────────────────────┬────────────────────────┬────────────┬────────────┬─────────────────────┬────────────────────────────┬──────────────┬────────────┐
│     client_hash_id      │     content_hash_id      │     keyword_hash_id      │     url_hash_id      │ keyword_char_count │ keyword_token_count │ url_char_count │ content_created_date │ content_updated_date │  content_type   │ search_volume │ competition │ competition_level │  cpc   │  main_intent  │ backlinks │ category_count │ keyword_created_date │      provider_used      │       model_used       │ char_count │ word_count │ last_optimized_date │ optimization_eligible_date │ is_p

In [15]:
result = con.sql(f"""
SELECT
  f.report_date,
  f.client_hash_id,
  f.content_hash_id,
  f.scroll_events,
  f.sessions_ai,
  f.sessions_referral,
  f.ga4_sessions / NULLIF(f.ga4_users, 0) AS sessions_per_user,
  CASE
    WHEN d.content_updated_date < f.report_date
    THEN f.report_date - d.content_updated_date
    ELSE NULL
  END AS days_since_update
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') AS f
JOIN read_parquet('{rel}/dim_content.parquet') AS d
  ON f.content_hash_id = d.content_hash_id
WHERE f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
  AND f.gsc_data_available = true
LIMIT 20
""")
result.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬───────────────┬─────────────┬───────────────────┬───────────────────┬───────────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ scroll_events │ sessions_ai │ sessions_referral │ sessions_per_user │ days_since_update │
│    date     │         varchar         │         varchar          │     int64     │    int64    │       int64       │      double       │       int64       │
├─────────────┼─────────────────────────┼──────────────────────────┼───────────────┼─────────────┼───────────────────┼───────────────────┼───────────────────┤
│ 2026-03-09  │ client_2094c6eb080311d5 │ content_14a3d47ccd0d15dc │             0 │           0 │                 0 │              NULL │              NULL │
│ 2026-03-31  │ client_2094c6eb080311d5 │ content_14a6f92117604fef │             0 │           0 │                 0 │              NULL │              NULL │
│ 2026-03-31  │ client_2094c6eb080311d5 │ cont

In [16]:
check = con.sql(f"""
SELECT
  COUNT(*) AS total,
  COUNT(f.ga4_users) AS has_ga4_users,
  COUNT(d.content_updated_date) AS has_update_date
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') AS f
JOIN read_parquet('{rel}/dim_content.parquet') AS d
  ON f.content_hash_id = d.content_hash_id
WHERE f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
  AND f.gsc_data_available = true
""")
check.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬───────────────┬─────────────────┐
│  total  │ has_ga4_users │ has_update_date │
│  int64  │     int64     │      int64      │
├─────────┼───────────────┼─────────────────┤
│ 3611061 │       2082695 │         3611061 │
└─────────┴───────────────┴─────────────────┘



In [17]:
timing_check = con.sql(f"""
SELECT
  COUNT(*) AS total,
  COUNT(CASE WHEN d.content_updated_date < f.report_date THEN 1 END) AS updated_before,
  COUNT(CASE WHEN d.content_updated_date >= f.report_date THEN 1 END) AS updated_after_or_same
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') AS f
JOIN read_parquet('{rel}/dim_content.parquet') AS d
  ON f.content_hash_id = d.content_hash_id
WHERE f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
  AND f.gsc_data_available = true
""")
timing_check.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────────┬───────────────────────┐
│  total  │ updated_before │ updated_after_or_same │
│  int64  │     int64      │         int64         │
├─────────┼────────────────┼───────────────────────┤
│ 3611061 │         648576 │               2962485 │
└─────────┴────────────────┴───────────────────────┘



Note on days_since_update: Only ~18% of rows (648,576 of 3,611,061) have a usable value for this feature in March 2026, since most content's content_updated_date falls after this month (the dataset appears to skew toward recent updates near the July 2026 export date). This feature remains valid and leakage-free, but its coverage is limited for this specific time window.